# Entrenamiento urbano en Google Colab Free

Notebook de preparación y ejecución controlada para la rama `urban-models-and-benchmark`. No descarga datasets automáticamente ni inicia entrenamientos al abrirlo.

**Importante:** Colab Free no garantiza GPU ni persistencia de la máquina. Los checkpoints se guardan en Google Drive; los datasets deben estar disponibles en el almacenamiento remoto de Colab mediante una fuente autorizada. Nunca descargues datasets al teléfono.


## 1. Configuración inicial
Ajusta la rama y las rutas si fuera necesario. Los datasets se ubican en el disco temporal de Colab (`/content/datasets`) para evitar lecturas lentas desde Drive. Los resultados se guardan en Drive.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json

REPO_URL = "https://github.com/Leohf04/Procesamiento-de-Im-genes-.git"
BRANCH = "urban-models-and-benchmark"
WORKDIR = Path("/content/Procesamiento-de-Im-genes-")
DATA_DIR = Path("/content/datasets")  # almacenamiento temporal remoto; no es el teléfono
DRIVE_RESULTS = Path("/content/drive/MyDrive/ProcesamientoImagenes/resultados")
BDD_ROOT = DATA_DIR / "bdd100k"

DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Configuración preparada")
print("Repositorio:", REPO_URL, "\nRama:", BRANCH)
print("Datos temporales:", DATA_DIR)
print("Resultados persistentes:", DRIVE_RESULTS)


## 2. Montar Google Drive
Autoriza el montaje desde la ventana de Google. Drive conserva resultados, no se usa como carpeta de entrenamiento de imágenes masivas.


**Permisos de acceso:** el token Fine-grained necesita `Contents: Read-only` para clonar el repo y `Actions: Read-only` para descargar el artifact privado existente. Si prefieres no crear un token con Actions read, puedes ejecutar las pruebas primero y omitir las celdas 5.5–5.6 hasta disponer de acceso.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
for folder in ["checkpoints", "metricas", "graficas", "informes", "logs"]:
    (DRIVE_RESULTS / folder).mkdir(parents=True, exist_ok=True)
print("Drive listo:", DRIVE_RESULTS)


## 3. Clonar el repositorio privado
Crea en **Colab → Secrets** un secreto llamado `GITHUB_TOKEN` con un Fine-grained Personal Access Token de GitHub limitado a este repositorio y con permiso **Contents: Read-only**. No escribas el token en una celda ni lo compartas en el chat. Si no está en Secrets, el notebook solicitará el token de forma oculta durante esta sesión.


In [ ]:
from getpass import getpass
try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = None
if not github_token:
    github_token = getpass("GitHub token (no se mostrará): ")
if not github_token:
    raise RuntimeError("Se necesita un token GitHub de solo lectura para clonar el repositorio privado.")

askpass = Path('/tmp/colab_git_askpass.sh')
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo x-access-token ;;\n  *Password*) echo "$COLAB_GITHUB_TOKEN" ;;\n  *) exit 1 ;;\nesac\n')
askpass.chmod(0o700)
env = os.environ.copy()
env.update({"GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0", "COLAB_GITHUB_TOKEN": github_token})
if WORKDIR.exists():
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", BRANCH], check=True, env=env)
    subprocess.run(["git", "-C", str(WORKDIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(WORKDIR), "pull", "--ff-only", "origin", BRANCH], check=True, env=env)
else:
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(WORKDIR)], check=True, env=env)
del github_token, env
print(subprocess.check_output(["git", "-C", str(WORKDIR), "log", "-1", "--oneline"], text=True))


## 4. Dependencias compatibles con Colab
No reinstalamos `torch` ni `torchvision`: Colab ya proporciona builds preparados para su entorno CUDA. Instalamos únicamente dependencias adicionales del proyecto.


In [ ]:
%cd /content/Procesamiento-de-Im-genes-
!python -m pip install -q "opencv-python-headless>=4.8,<5" "scikit-learn>=1.3" "matplotlib>=3.7" "PyYAML>=6" "tqdm>=4.66" "joblib>=1.3" "onnx>=1.14" "onnxruntime>=1.17" "Pillow>=9" "pycocotools>=2.0.7" "pytest>=7"


## 5. Diagnóstico del entorno
Si `torchvision` no importa o no coincide con PyTorch, detén el proceso y comparte el error; no actualices CUDA/PyTorch a ciegas.


## 5.3. Pruebas multidataset y alineación de sample_id
Estas pruebas cubren A2D2, UrbanSyn, adaptadores semánticos/panoptic, targets de instancias, balanceo, alineación estricta por `sample_id`, CLI del planificador e indexador. No descargan imágenes ni entrenan modelos.

In [ ]:
import os, subprocess, sys
os.chdir(WORKDIR)
TARGETED_TESTS = [
    "tests/test_multidataset_adapters.py",
    "tests/test_urban_training_loader.py",
    "tests/test_balanced_training_plan_cli.py",
    "tests/test_remote_annotation_index.py",
]
result = subprocess.run([sys.executable, "-m", "pytest", "-q", *TARGETED_TESTS], text=True)
print("Código de salida:", result.returncode)
if result.returncode:
    raise RuntimeError("Fallaron las pruebas multidataset; no continuar al plan de balanceo hasta corregirlas.")

## 5.4. Suite completa de pruebas
Ejecuta la suite del repositorio después de que las pruebas críticas anteriores pasen. Si hay fallos, conserva la salida completa y detén el flujo; no se inicia entrenamiento.

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q"], text=True)
print("Código de salida:", result.returncode)
if result.returncode:
    raise RuntimeError("La suite completa no pasó; detener y revisar los fallos.")

## 5.5. Recuperar el artifact de anotaciones existente (sin reindexar)
El artifact `14770171764` del run `45038801547` contiene los seis índices JSONL y los manifiestos enriquecidos del plan anterior. Para descargarlo desde el repositorio privado, el secreto `GITHUB_TOKEN` debe ser un Fine-grained PAT limitado a este repositorio con permisos **Contents: Read-only** y **Actions: Read-only**. No pegues el token en una celda ni en el chat. Esta celda descarga únicamente el ZIP del artifact ya existente; no vuelve a consultar S3/Hugging Face ni descarga imágenes/máscaras.

In [ ]:
import os, zipfile, requests, json, shutil
from pathlib import Path
from getpass import getpass
try:
    from google.colab import userdata
    artifact_token = userdata.get("GITHUB_TOKEN")
except Exception:
    artifact_token = None
if not artifact_token:
    artifact_token = getpass("GitHub token (Contents: read + Actions: read; oculto): ")
if not artifact_token:
    raise RuntimeError("Falta token de solo lectura con Actions: read para recuperar el artifact privado.")
ARTIFACT_ID = 14770171764
ARTIFACT_ZIP = DATA_DIR / "urban-index-artifact-14770171764.zip"
ARTIFACT_DIR = DATA_DIR / "urban-index-artifact-14770171764"
headers = {"Authorization": f"Bearer {artifact_token}", "Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28"}
url = f"https://api.github.com/repos/Leohf04/Procesamiento-de-Im-genes-/actions/artifacts/{ARTIFACT_ID}/zip"
response = requests.get(url, headers=headers, stream=True, timeout=180)
if response.status_code != 200:
    raise RuntimeError(f"No se pudo descargar artifact {ARTIFACT_ID}: HTTP {response.status_code}. Verifica Actions: read y que el artifact no haya expirado.")
with ARTIFACT_ZIP.open("wb") as handle:
    for chunk in response.iter_content(chunk_size=1024 * 1024):
        if chunk:
            handle.write(chunk)
with zipfile.ZipFile(ARTIFACT_ZIP) as archive:
    root = ARTIFACT_DIR.resolve()
    for member in archive.infolist():
        destination = (ARTIFACT_DIR / member.filename).resolve()
        if root not in destination.parents and destination != root:
            raise RuntimeError(f"Ruta insegura dentro del ZIP: {member.filename}")
    archive.extractall(ARTIFACT_DIR)
del artifact_token, headers, response
print("Artifact recuperado:", ARTIFACT_ZIP.stat().st_size, "bytes")
for path in ARTIFACT_DIR.rglob("*.jsonl"):
    print(path.relative_to(ARTIFACT_DIR))

## 5.6. Reconstruir manifiestos base desde el artifact y generar plan con cap 200
El artifact existente conserva los índices de anotaciones y los JSONL enriquecidos del plan previo. Se reconstruyen los manifiestos de entrada desde `*_semantic_{split}.jsonl` (que contienen los registros base más estadísticas), se valida que estén los seis índices y seis manifiestos fuente, y se ejecuta exclusivamente el planificador local con `per_class_cap=200`. No se ejecuta `index_remote_annotations.py` ni se descargan anotaciones nuevas. El cap limita la presencia de cada clase por número de muestras, no iguala píxeles ni número de objetos.

In [ ]:
import json, subprocess, sys
from pathlib import Path
candidates = list(ARTIFACT_DIR.rglob("*.annotations.jsonl"))
if not candidates:
    raise FileNotFoundError("El artifact no contiene índices *.annotations.jsonl; detener sin reindexar.")
annotation_dir = candidates[0].parent
required_indexes = [annotation_dir / f"{ds}_{split}.annotations.jsonl" for ds in ("a2d2", "urbansyn") for split in ("train", "val", "test")]
missing = [str(p) for p in required_indexes if not p.is_file()]
if missing:
    raise FileNotFoundError("Faltan índices existentes: " + ", ".join(missing))
manifest_dir = DATA_DIR / "reconstructed_manifests"
manifest_dir.mkdir(parents=True, exist_ok=True)
for ds in ("a2d2", "urbansyn"):
    for split in ("train", "val", "test"):
        source = next(ARTIFACT_DIR.rglob(f"{ds}_semantic_{split}.jsonl"), None)
        if source is None:
            raise FileNotFoundError(f"No se encontró manifiesto enriquecido {ds}_semantic_{split}.jsonl")
        rows = [json.loads(line) for line in source.read_text(encoding="utf-8").splitlines() if line.strip()]
        if not rows or any(not row.get("sample_id") for row in rows):
            raise ValueError(f"Manifiesto fuente vacío o con sample_id inválido: {source}")
        destination = manifest_dir / f"{ds}_{split}.jsonl"
        destination.write_text("".join(json.dumps(row, sort_keys=True) + "\n" for row in rows), encoding="utf-8")
output_dir = DATA_DIR / "balanced_cap200"
command = [sys.executable, "scripts/build_balanced_training_plan.py", "--manifests-dir", str(manifest_dir), "--annotation-index-dir", str(annotation_dir), "--output-dir", str(output_dir), "--seed", "2026", "--max-samples-per-split", "1000", "--per-class-cap", "200"]
subprocess.run(command, cwd=WORKDIR, check=True)
report_path = output_dir / "balanced_training_plan.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print("Plan generado:", report_path)
print(json.dumps({"per_class_cap": report["per_class_cap"], "plans": report["plans"]}, indent=2)[:20000])
(DRIVE_RESULTS / "informes").mkdir(parents=True, exist_ok=True)
shutil.copy2(report_path, DRIVE_RESULTS / "informes" / "balanced_training_plan_cap200.json")
print("Informe copiado a Drive:", DRIVE_RESULTS / "informes" / "balanced_training_plan_cap200.json")

## 5.7. Guardar evidencia de ejecución en Drive
Guarda el SHA de Git, versiones, resultados y plan. No copies tokens ni datasets. Ejecuta esta celda al terminar una etapa satisfactoria.

In [ ]:
import subprocess, json, sys, torch
from pathlib import Path
log = {
    "git_head": subprocess.check_output(["git", "-C", str(WORKDIR), "rev-parse", "HEAD"], text=True).strip(),
    "git_branch": subprocess.check_output(["git", "-C", str(WORKDIR), "branch", "--show-current"], text=True).strip(),
    "python": sys.version,
    "torch": torch.__version__,
    "torchvision": __import__("torchvision").__version__,
    "cuda_available": torch.cuda.is_available(),
    "run_training": False,
}
if torch.cuda.is_available():
    log["gpu"] = torch.cuda.get_device_name(0)
path = DRIVE_RESULTS / "informes" / "colab_validation_environment.json"
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(log, indent=2) + "\n", encoding="utf-8")
print(json.dumps(log, indent=2))
print("Evidencia:", path)

In [ ]:
import torch, torchvision
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    !nvidia-smi
else:
    print("AVISO: esta sesión no recibió GPU. No inicies entrenamiento pesado.")


In [ ]:
import subprocess
subprocess.run([sys.executable, "scripts/verify_setup.py"], check=True)


## 5.1. Pruebas sintéticas de los adaptadores BDD100K
Estas pruebas usan imágenes/máscaras temporales pequeñas; no requieren descargar BDD100K ni activan entrenamiento.


In [ ]:
!python -m pytest -q tests/test_bdd100k_dataset.py
!python scripts/train_urban_segmentation.py --help
!python scripts/train_maskrcnn_cityscapes.py --help


## 5.2. Validación con muestras reales de BDD100K (solo semántica)
Esta prueba obtiene únicamente 2 imágenes y sus máscaras de train y 2 de validación desde `dronefreak/BDD100K-Semantic-Segmentation`. Empareja `data/images_10k/10k/{train,val}` con `data/seg_maps/labels/{train,val}` por nombre de archivo (`_train_id.png`), sin depender de metadata.jsonl. No descarga el paquete completo, no usa Drive para datasets y no entrena. El script verifica dimensiones, IDs 0–18/255 y ejecuta el loader real. La copia es una redistribución de terceros: resultado provisional, no certificación oficial ni validación de instancias/detección/carriles.


In [ ]:
!python scripts/validate_bdd100k_semantic_sample.py --samples-per-split 2


## 6. Comprobar datasets (sin descargarlos)
BDD100K requiere imágenes 10K y máscaras oficiales sem_seg/ins_seg en train/val. Esta celda solo verifica estructura; no descarga datos. Los datos deben obtenerse desde una fuente oficial/autorizada directamente en `/content`, nunca desde el teléfono.


In [ ]:
def count_bdd100k():
    result = {}
    images_root = BDD_ROOT / "images" / "10k"
    for split in ("train", "val"):
        images = list((images_root / split).glob("*.jpg"))
        semantic = list((BDD_ROOT / "labels" / "sem_seg" / "masks" / split).glob("*.png"))
        instances = list((BDD_ROOT / "labels" / "ins_seg" / "bitmasks" / split).glob("*.png"))
        result[split] = {"images": len(images), "semantic_masks": len(semantic),
                         "instance_bitmasks": len(instances)}
    return result

if BDD_ROOT.exists():
    print(json.dumps(count_bdd100k(), indent=2))
else:
    print("BDD100K no está en", BDD_ROOT,
          "— pendiente de acceso/descarga autorizada directa en la VM.")


## 6.1. Auditoría obligatoria de instancias BDD100K oficial
Antes de entrenar Mask R-CNN, ejecuta esta auditoría sobre las bitmasks RGBA oficiales. Comprueba emparejamiento exacto, dimensiones, IDs/flags, separación train/val y cobertura de las ocho clases. La muestra pública de GitHub (2 imágenes, 4 clases, split val) **no sirve como conjunto de entrenamiento** y no se utiliza aquí. El informe se guarda en el disco temporal de Colab; no se descarga nada al teléfono.


In [ ]:
INSTANCE_PREFLIGHT = DATA_DIR / "bdd100k_instance_preflight.json"
if all((BDD_ROOT / "images" / "10k" / split).is_dir() and
       (BDD_ROOT / "labels" / "ins_seg" / "bitmasks" / split).is_dir()
       for split in ("train", "val")):
    subprocess.run([
        sys.executable, "scripts/audit_bdd100k_instance_splits.py",
        "--train-images", str(BDD_ROOT / "images" / "10k" / "train"),
        "--train-bitmasks", str(BDD_ROOT / "labels" / "ins_seg" / "bitmasks" / "train"),
        "--val-images", str(BDD_ROOT / "images" / "10k" / "val"),
        "--val-bitmasks", str(BDD_ROOT / "labels" / "ins_seg" / "bitmasks" / "val"),
        "--output", str(INSTANCE_PREFLIGHT),
    ], check=True)
else:
    print("Auditoría pendiente: faltan los directorios oficiales de imágenes/bitmasks train y val.")


## 7. Configurar entrenamiento
No ejecutes esta sección hasta tener BDD100K autorizado, completo y validado. El valor `RUN_TRAINING` permanece desactivado deliberadamente. Los scripts ahora guardan un checkpoint reanudable `.last.pt` y aceptan `--resume`.

Para reanudar una ejecución interrumpida, conserva la misma ruta de checkpoint y añade `--resume`. `--epochs` representa el total final de épocas, no las épocas adicionales.


In [ ]:
RUN_TRAINING = False  # Mantener False hasta validar datos y smoke tests
MODEL = "semantic"  # "semantic" o "instance"
EPOCHS = 30
BATCH_SIZE = 2
WORKERS = 2
RESUME = False

BDD_IMAGES = BDD_ROOT / "images" / "10k"
BDD_SEMANTIC = BDD_ROOT / "labels" / "sem_seg" / "masks"
BDD_INSTANCE = BDD_ROOT / "labels" / "ins_seg" / "bitmasks"

def bdd_split_ready(label_root, split):
    images = list((BDD_IMAGES / split).glob("*.jpg"))
    masks = list((label_root / split).glob("*.png"))
    return len(images), len(masks)

print("BDD100K semantic train:", bdd_split_ready(BDD_SEMANTIC, "train"))
print("BDD100K semantic val:", bdd_split_ready(BDD_SEMANTIC, "val"))
print("BDD100K instance train:", bdd_split_ready(BDD_INSTANCE, "train"))
print("BDD100K instance val:", bdd_split_ready(BDD_INSTANCE, "val"))

if RUN_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError("No hay GPU disponible; se cancela el entrenamiento.")
    if MODEL == "semantic":
        if not all((BDD_IMAGES / split).is_dir() and (BDD_SEMANTIC / split).is_dir()
                   for split in ("train", "val")):
            raise FileNotFoundError("Faltan imágenes o máscaras sem_seg de BDD100K en train/val.")
        output = DRIVE_RESULTS / "checkpoints" / "deeplabv3plus_bdd100k_semantic.pt"
        command = [sys.executable, "scripts/train_urban_segmentation.py",
                   "--task", "bdd100k-semantic",
                   "--images-dir", str(BDD_IMAGES / "train"),
                   "--masks-dir", str(BDD_SEMANTIC / "train"),
                   "--val-images-dir", str(BDD_IMAGES / "val"),
                   "--val-masks-dir", str(BDD_SEMANTIC / "val"),
                   "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
                   "--workers", str(WORKERS), "--output", str(output)]
    elif MODEL == "instance":
        if not INSTANCE_PREFLIGHT.is_file():
            raise RuntimeError("Ejecuta primero la auditoría oficial de instancias (sección 6.1).")
        preflight = json.loads(INSTANCE_PREFLIGHT.read_text(encoding="utf-8"))
        if not preflight.get("ready_for_training"):
            raise RuntimeError("La auditoría oficial no aprobó train/val o faltan clases; entrenamiento cancelado.")
        if not all((BDD_IMAGES / split).is_dir() and (BDD_INSTANCE / split).is_dir()
                   for split in ("train", "val")):
            raise FileNotFoundError("Faltan imágenes o bitmasks ins_seg de BDD100K en train/val.")
        output = DRIVE_RESULTS / "checkpoints" / "maskrcnn_bdd100k.pt"
        command = [sys.executable, "scripts/train_maskrcnn_cityscapes.py",
                   "--dataset", "bdd100k",
                   "--images-dir", str(BDD_IMAGES / "train"),
                   "--bitmasks-dir", str(BDD_INSTANCE / "train"),
                   "--val-images-dir", str(BDD_IMAGES / "val"),
                   "--val-bitmasks-dir", str(BDD_INSTANCE / "val"),
                   "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
                   "--workers", str(WORKERS), "--output", str(output)]
    else:
        raise ValueError("MODEL debe ser 'semantic' o 'instance'.")
    if RESUME:
        command.append("--resume")
    subprocess.run(command, check=True)
else:
    print("Entrenamiento desactivado. No se iniciará hasta validar datasets y GPU.")


## 8. Evaluación y persistencia
Después de entrenar, ejecutaremos los evaluadores existentes y guardaremos métricas e informes en `DRIVE_RESULTS`. No subas datasets ni checkpoints grandes al repositorio GitHub. Los artefactos grandes permanecen en Drive; el código y reportes ligeros permanecen en GitHub.

**Al terminar cada sesión:** confirma que los archivos `.pt`, `.last.pt` y `.history.json` existen en Drive antes de desconectar. Si la sesión se interrumpe, monta Drive, clona/actualiza el repo y reanuda con `--resume`.
